# Introduction

We are now moving to the final part of the workshop, which involves formulating business recommendations. Our tasks are:
- Determining a global betting odds,
- Dividing the dataset into categories: A, B, C, D, where A is the best group and D is the weakest group,
- Determining the risk of odds based on accepted parameters for each category.

As the last task, in a discussion format, we must consider the fact that we are a new betting company. When formulating our recommendations, we need to identify the risks that may affect our operations. We will perform this task together in a brainstorming session.

# Notebook Configuration

## Import necessary libraries

In [142]:
import pandas as pd
import numpy as np


## Loading data into the workspace

> Remember to correctly specify the column separator

In [111]:
df = pd.read_csv("../data/processed/hockey_teams.csv", sep=";")

### Checking data loading accuracy

In [120]:
global_odds = round(1 / df["victory_percentage"].mean(), 2)

print("Globální kurz:",global_odds)
df["victory_percentage"].describe()

Globální kurz: 2.18


count    582.000000
mean       0.458519
std        0.102237
min        0.119000
25%        0.390000
50%        0.463000
75%        0.534000
max        0.756000
Name: victory_percentage, dtype: float64

# Determining Betting Odds

Let's review the content of the page: [click](https://trustbet.pl/kursy-bukmacherskie/), where information about methods for determining betting odds can be found. First, we will determine a global odd, which will be the starting point for our analysis (the so-called _baseline scenario_). At this point, we ignore the margin and assume that we are calculating the decimal odd.

Here is the list of steps to be performed to obtain the desired value:
- we will complete the definition of the `get_betting_odds` function, which will take `probability` of a given event as a parameter. We will use it multiple times, so it is worth preparing its implementation now
- then we need to appropriately aggregate the set and determine the **global** probability of the team's victory.

## Implementations of the `get_betting_odds` function

In [143]:
def get_betting_odds(probability):
    """Vrátí sázkový kurz pro danou pravděpodobnost výhry."""
    if probability <= 0:
        raise ZeroDivisionError("Pravděpodobnost musí být větší než 0")
    return round(1 / probability, 2)

### Some tests to check the correctness of the implementation

In [144]:
def test_get_betting_odds():
    assert get_betting_odds(1) == 1, "Expected 1"
    assert get_betting_odds(0.5) == 2, "Expected 2"
    assert get_betting_odds(0.25) == 4, "Expected 4"
    assert get_betting_odds(0.1) == 10, "Expected 10"
    try:
        get_betting_odds(0)
    except ZeroDivisionError:
        pass
    else:
        assert False, "Expected ZeroDivisionError"

    print("All tests passed!")

test_get_betting_odds()

All tests passed!


### Determining the global odds

Here, determine the probability of any team winning

In [130]:
# globální pravděpodobnost výhry = průměr všech týmů
global_probability = df["victory_percentage"].mean()
print("Globální pravděpodobnost výhry:", round(global_probability, 3))

#mean() spočítá průměr napříč všemi týmy a sezónami.

Globální pravděpodobnost výhry: 0.459


Set the global rate here using the `get_betting_odds` function. Round the result to two decimal places.

In [131]:
# globální kurz podle průměrné pravděpodobnosti výhry
global_odd = round(get_betting_odds(global_probability), 2)

print("Globální sázkový kurz:", global_odd)

Globální sázkový kurz: 2.18


# Team Categorization

Let's discuss how we can classify teams into _leagues_. We want to establish 4 leagues:
- A - league consisting of the best teams,
- B - league consisting of good teams,
- C - league consisting of average teams,
- D - league consisting of the weakest teams.

The above terms are quite subjective, so for the purpose of this exercise, we will adopt the following assumptions:
- A - the top 5% of teams,
- B - teams performing better than 70% of the group but worse than league A,
- C - teams performing better than 20% of the group but worse than league B,
- D - the remaining teams.

To accomplish this task, we will additionally implement the function `assign_team_to_league`.

> Note: This task looks unassuming, but it is difficult. Remember that during the class, you have access to the instructor, and later to a mentor.

## Determination of cutoff points for individual leagues

In [145]:
def assign_team_to_league(team_value, all_values):
    """Vrátí ligu A-D podle výkonu týmu."""
    top_5 = np.percentile(all_values, 95)  # horních 5 %
    top_30 = np.percentile(all_values, 70) # 70 % hranice
    top_20 = np.percentile(all_values, 20) # 20 % hranice

    if team_value >= top_5:
        return "A"
    elif team_value >= top_30:
        return "B"
    elif team_value >= top_20:
        return "C"
    else:
        return "D"


In [ ]:
# přiřazení lig podle victory_percentage
df["league"] = df["victory_percentage"].apply(lambda x: assign_team_to_league(x, df["victory_percentage"]))

# zkontrolujeme, že sloupec vznikl
print(df[["Team", "victory_percentage", "league"]].head())


## Determination of odds per league

Here we set the betting odds for each league, which will allow us to draw final conclusions and establish the basic odds for individual teams.

> Remember: After generating the results, it is worth checking if they are reasonable.

In [149]:
# uložíme všechny victory_percentage do pole
all_values = df["victory_percentage"].values

# funkce pro přiřazení ligy
def assign_team_to_league(team_value):
    top_5 = np.percentile(all_values, 95)
    top_30 = np.percentile(all_values, 70)
    top_20 = np.percentile(all_values, 20)
    
    if team_value >= top_5:
        return "A"
    elif team_value >= top_30:
        return "B"
    elif team_value >= top_20:
        return "C"
    else:
        return "D"

# vytvoření sloupce league
df["league"] = df["victory_percentage"].apply(assign_team_to_league)

# slovník pro uložení kurzů jednotlivých lig
league_odds = {}
for league in df["league"].unique():
    league_probs = df.loc[df["league"] == league, "victory_percentage"]
    avg_prob = league_probs.mean()
    league_odds[league] = get_betting_odds(avg_prob)

# zobrazíme výsledky
for league, odd in league_odds.items():
    print(f"Kurzy ligy {league}: {odd}")


Kurzy ligy B: 1.81
Kurzy ligy C: 2.27
Kurzy ligy D: 3.33
Kurzy ligy A: 1.56


# Discussion

We have obtained certain odds values for each league. But how does this translate into real business? The entire task was about determining certain values from which a bookmaker can begin operations. Correct determination of these values is critical to attract customers to place bets with us, and on the other hand, inappropriate determination may lead to financial losses in the first days of operation.

For this reason, before translating the results and recommendations into business objectives, the analysis is subjected to discussion. Therefore, we will now take on a review role and would like to verify the steps. To that end, we will collectively discuss and critique our work by answering the following questions together:
- What elements of the analysis were simplified? What was omitted in the analysis?
- Are there any inconsistencies in the estimated odds? What are they?
- How can we improve the odds estimates?
- How can we enrich our initial dataset to make the estimates more accurate and less risky?
- How can we simulate the outcomes of our analysis to verify that they do not lead to financial losses?

This is a discussion panel, and every idea is valuable here.


In [ ]:
#Rozřadili jsme týmy do lig A–D a spočítali základní sázkové kurzy.

# Bohuzel odhady nezohledňují všechny faktory jako je forma hrace a treba prestupy do jinych tymu, v propade nemoci nebo zraneni se mohou 
# vysledky vyvyjet take jinak. 

# Pro reálný byznys je potřeba data rozšířit a simulovat výsledky, aby se snížilo riziko ztrát a taky to musi delta nekdo kdo s tim ma nejlepe 
 # zkusenosti, take potrebujeme nekho kdo ty zapasy sleduje a dokaze lepe rozenat kam se tymy posunuji